# Phase 1 Task 2 — Pre-Project Feature Scope & Profiling
## 1. Task objective
Scope a binary prediction problem, audit features and leakage, quantify balance, establish reproducible splits and evaluate a trivial baseline.

## 2. Imports / reproducibility

In [1]:
from src.load_data import load_dataset, predictors_and_target, TARGET
from src.profile_features import profile_features, duplicate_row_count
from src.leakage_check import suspicious_feature_names, exact_target_copies, vetted_features
from src.split_data import RANDOM_STATE, split_data, split_summary
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score, f1_score, precision_score, roc_auc_score, accuracy_score
RANDOM_STATE

42

## 3. Data loading
The official brief did not specify a dataset, so the scikit-learn Wisconsin Breast Cancer dataset is used as a reproducible real-data demonstration.

In [2]:
df = load_dataset()
X, y = predictors_and_target(df)
df.shape, X.shape

((569, 31), (569, 30))

## 4. Problem definition
Given 30 quantitative features computed from a digitized fine-needle aspirate image, predict whether each breast-mass sample is malignant or benign. One row is one diagnostic sample.

## 5. Target definition
`target`: 0 = malignant (evaluation-positive/critical class), 1 = benign.

In [3]:
y.value_counts().rename(index={0: 'malignant', 1: 'benign'})

target
benign       357
malignant    212
Name: count, dtype: int64

## 6. Manual label examples

In [4]:
df.loc[[0, 1, 19, 20, 100, 200, 300, 500], ['mean radius', 'mean texture', TARGET]].assign(label=lambda d: d[TARGET].map({0:'malignant',1:'benign'}))

,mean radius,mean texture,target,label
sample_id,,,,
0,17.99,10.38,0,malignant
1,20.57,17.77,0,malignant
19,13.54,14.36,1,benign
20,13.08,15.71,1,benign
100,13.61,24.98,0,malignant
200,12.23,19.56,1,benign
300,19.53,18.90,0,malignant
500,15.04,16.74,1,benign


## 7. Feature inventory

In [5]:
profile = profile_features(X)
profile[['feature','dtype','missing_count','unique_count','candidate_status']]

,feature,dtype,missing_count,unique_count,candidate_status
0,mean radius,float64,0,456,KEEP
1,mean texture,float64,0,479,KEEP
2,mean perimeter,float64,0,522,KEEP
3,mean area,float64,0,539,KEEP
4,mean smoothness,float64,0,474,KEEP
5,mean compactness,float64,0,537,KEEP
6,mean concavity,float64,0,537,KEEP
7,mean concave points,float64,0,542,KEEP
8,mean symmetry,float64,0,432,KEEP
9,mean fractal dimension,float64,0,499,KEEP


## 8. Missing-value analysis

In [6]:
{'missing_values': int(X.isna().sum().sum()), 'duplicate_complete_rows': duplicate_row_count(df)}

{'missing_values': 0, 'duplicate_complete_rows': 0}

## 9. Leakage audit

In [7]:
{'target_in_X': TARGET in X, 'suspicious_names': suspicious_feature_names(list(X.columns)), 'target_copies': exact_target_copies(X, y)}

{'target_in_X': False, 'suspicious_names': [], 'target_copies': []}

## 10. Class balance
## 11. Base rate

In [8]:
balance = y.value_counts().sort_index().to_frame('count')
balance['percentage'] = y.value_counts(normalize=True).sort_index() * 100
balance.rename(index={0:'malignant',1:'benign'})

,count,percentage
target,,
malignant,212,37.258348
benign,357,62.741652


## 12. Vetted feature list

In [9]:
vetted = vetted_features(X, profile, TARGET)
len(vetted), vetted

(30,
 ['mean radius',
  'mean texture',
  'mean perimeter',
  'mean area',
  'mean smoothness',
  'mean compactness',
  'mean concavity',
  'mean concave points',
  'mean symmetry',
  'mean fractal dimension',
  'radius error',
  'texture error',
  'perimeter error',
  'area error',
  'smoothness error',
  'compactness error',
  'concavity error',
  'concave points error',
  'symmetry error',
  'fractal dimension error',
  'worst radius',
  'worst texture',
  'worst perimeter',
  'worst area',
  'worst smoothness',
  'worst compactness',
  'worst concavity',
  'worst concave points',
  'worst symmetry',
  'worst fractal dimension'])

## 13. Train/validation/test split

In [10]:
splits = split_data(X[vetted], y)
split_summary(splits)

,split,rows,malignant,benign
0,train,398,148,250
1,validation,85,32,53
2,test,86,32,54


## 14. Baseline

In [11]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE).fit(splits.X_train, splits.y_train)
pred = dummy.predict(splits.X_test)
p_malignant = dummy.predict_proba(splits.X_test)[:, list(dummy.classes_).index(0)]
{'malignant_recall': recall_score(splits.y_test,pred,pos_label=0,zero_division=0), 'malignant_f1': f1_score(splits.y_test,pred,pos_label=0,zero_division=0), 'malignant_precision': precision_score(splits.y_test,pred,pos_label=0,zero_division=0), 'roc_auc': roc_auc_score((splits.y_test==0).astype(int),p_malignant), 'accuracy': accuracy_score(splits.y_test,pred)}

{'malignant_recall': 0.0,
 'malignant_f1': 0.0,
 'malignant_precision': 0.0,
 'roc_auc': 0.5,
 'accuracy': 0.627906976744186}

## 15. Go/no-go recommendation
**GO** for controlled baseline modelling, not clinical deployment. The target is clear, all 30 predictors are usable, leakage is controlled, missingness is absent and moderate imbalance is manageable.

## 16. Final summary
Use training-fitted standardization with class-weighted logistic regression as the first interpretable candidate, then compare a random forest. Select thresholds on validation data and evaluate once on the untouched test set.